In [1]:
from arcgis.gis import GIS
from datetime import datetime
from datetime import timezone
now = datetime.now(timezone.utc)

In [2]:
gis_premium = GIS("https://www.arcgis.com", "hubpy_test", "hubPython01")
myhub = gis_premium.hub
gis_basic = GIS("https://www.arcgis.com", "geosaurus_hub", "hubPython01")
basic_hub = gis_basic.hub
gis_portal = GIS("https://rpubs22001.ags.esri.com/portal/home/", "creator1", "portalaccount1")

### Add initiative

In [3]:
#Add initiative
title = "Test initiative %s" %int(now.timestamp() * 1000)
new_initiative = myhub.initiatives.add(title=title)
initiative_id = new_initiative.itemid
assert new_initiative.title==title, new_initiative.title

### Search for added initiative

In [4]:
#Searching for initiative
searched = myhub.initiatives.search(title=title, owner=gis_premium.users.me.username)
assert searched[0].itemid==new_initiative.itemid, searched.item

### Get initiative

In [5]:
#Fetching initiative
fetched = myhub.initiatives.get(initiative_id)
assert fetched==searched[0], fetched

### Update initiative

In [6]:
#Updating initiative
assert new_initiative.tags==[]
new_initiative.update(initiative_properties={'tags': 'Hub, OpenData'})
assert 'Hub' in new_initiative.tags, new_initiative.tags

### Clone initiative in same org, Basic org, and Enterprise deployment

In [7]:
def collab_group_exists(initiative):
    '''
    Verifies if collab group does not exist for this initiative/site
    '''
    if initiative.collab_group_id is None:
        return 'Works as expected'
    
def follower_group_exists(initiative):
    '''
    Verifies if follower group does not exist for this initiative/site
    '''
    try:
        initiative.followers_group_id
    except:
        return 'Works as expected'

In [8]:
#Cloning in the same organization
title = "Cloned initiative %s" %int(now.timestamp() * 1000)
cloned_initiative = myhub.initiatives.clone(new_initiative, title=title)
initiative_id = cloned_initiative.itemid
assert cloned_initiative.title==title, cloned_initiative.title
assert cloned_initiative.site_id
assert cloned_initiative.content_group_id
assert cloned_initiative.collab_group_id
assert cloned_initiative.followers_group_id

In [9]:
#Cloning in the Hub basic organization (without admin credentials)
title = "Cloned initiative %s" %int(now.timestamp() * 1000)
cloned_basic_initiative = basic_hub.initiatives.clone(new_initiative, origin_hub=myhub, title=title)
initiative_id = cloned_basic_initiative.itemid
assert cloned_basic_initiative.title==title, cloned_basic_initiative.title
assert collab_group_exists(cloned_basic_initiative)=='Works as expected', 'Collab group exists'
assert follower_group_exists(cloned_basic_initiative)=='Works as expected', 'Followers group exists'

In [10]:
#Cloning in the Exterprise portal (without admin credentials)
title = "Cloned site %s" %int(now.timestamp() * 1000)
site = myhub.sites.get(new_initiative.site_id)
cloned_site = gis_portal.sites.clone(site, title=title)
site_id = cloned_site.itemid
assert cloned_site.title==title, cloned_site.title

### Delete initiatives

In [11]:
def verify_deleted_group(gis, g_id):
    '''
    Verify group is deleted
    '''
    if gis.groups.get(g_id) is None:
        return 'Works as expected'
    

def verify_deleted_item(gis, i_id):
    '''
    Verify item is deleted
    '''
    if gis.content.get(i_id) is None:
        return 'Works as expected'

In [12]:
#Delete premium initiative
site_id = new_initiative.site_id
initiative_id = new_initiative.itemid
collab_group_id = new_initiative.collab_group_id
content_group_id = new_initiative.content_group_id
followers_group_id = new_initiative.followers_group_id
new_initiative.delete()
assert verify_deleted_item(gis_premium, initiative_id)=='Works as expected', 'initiative exists'
assert verify_deleted_item(gis_premium, site_id)=='Works as expected', 'site exists'
assert verify_deleted_group(gis_premium, collab_group_id)=='Works as expected', 'collab group exists'
assert verify_deleted_group(gis_premium, content_group_id)=='Works as expected', 'content group exists'
assert verify_deleted_group(gis_premium, followers_group_id)=='Works as expected', 'followers group exists'

In [13]:
#Delete premium cloned initiative
site_id = cloned_initiative.site_id
initiative_id = cloned_initiative.itemid
collab_group_id = cloned_initiative.collab_group_id
content_group_id = cloned_initiative.content_group_id
followers_group_id = cloned_initiative.followers_group_id
cloned_initiative.delete()
assert verify_deleted_item(gis_premium, initiative_id)=='Works as expected', 'initiative exists'
assert verify_deleted_item(gis_premium, site_id)=='Works as expected', 'site exists'
assert verify_deleted_group(gis_premium, collab_group_id)=='Works as expected', 'collab group exists'
assert verify_deleted_group(gis_premium, content_group_id)=='Works as expected', 'content group exists'
assert verify_deleted_group(gis_premium, followers_group_id)=='Works as expected', 'followers group exists'

In [14]:
#Delete basic cloned initiative
site_id = cloned_basic_initiative.site_id
initiative_id = cloned_basic_initiative.itemid
content_group_id = cloned_basic_initiative.content_group_id
cloned_basic_initiative.delete()
assert verify_deleted_item(gis_basic, initiative_id)=='Works as expected', 'initiative exists'
assert verify_deleted_item(gis_basic, site_id)=='Works as expected', 'site exists'
assert verify_deleted_group(gis_basic, content_group_id)=='Works as expected', 'content group exists'

In [15]:
#Delete enterprise cloned site
site_id = cloned_site.itemid
content_group_id = cloned_site.content_group_id
cloned_site.delete()
assert verify_deleted_item(gis_portal, site_id)=='Works as expected', 'site exists'
assert verify_deleted_group(gis_portal, content_group_id)=='Works as expected', 'content group exists'